# Try the Cyber-Fraud Complaint Router
Run a complaint through the trained Llama 3.1 8B LoRA model in your own Colab session. No paid inference API is used.

1. Choose **Runtime → Change runtime type → T4 GPU**, then connect. Free GPU availability is not guaranteed.
2. Run the cells below in order. Use your own Hugging Face account with approved [Llama access](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct) and a read token.
3. Initial model download is about 16 GB and can take 10–20 minutes. After loading, use the complaint box repeatedly.

Use synthetic or anonymized examples. This classroom prototype routes allegations; it does not verify crimes or provide calibrated confidence. Evaluation: 82/90 correct on synthetic test cases; human-review recall was 60%.

This demo reuses the loading procedure from the [executed evaluation notebook](https://github.com/deepdive-ai/cyber-fraud-router/blob/main/Cyber_Fraud_Fine_Tuning.ipynb).

In [ ]:
# Install before importing model libraries. Keep Colab's supplied CUDA-enabled PyTorch.
%pip install -q "transformers==5.17.0" "peft==0.20.0" "accelerate==1.15.0" "bitsandbytes==0.50.2" safetensors huggingface_hub ipywidgets


## Authenticate
Paste the token only into the hidden input below. If access is denied, confirm Llama access and token permissions on the same account.

In [ ]:
import torch
from huggingface_hub import login
from getpass import getpass
assert torch.cuda.is_available(), "Select a GPU runtime before continuing."
print("GPU:", torch.cuda.get_device_name(0))
login(token=getpass("Hugging Face read token: "), add_to_git_credential=False)


## Load the trained model
Assets are downloaded automatically from a fixed repository commit and checked for integrity. Run this cell once per session. Keep it running until it says the adapter loaded successfully.

In [ ]:
import hashlib
import json
from pathlib import Path
from urllib.request import urlopen

ASSET_URL = "https://raw.githubusercontent.com/deepdive-ai/cyber-fraud-router/e95e1c0c7b8333298efd1bb09d1336d8074320b6/"
ASSET_HASHES = {'adapter/adapter_config.json': 'f3d9a7a6673b8f0f264ab70fcb652e0b72c901f2ce178271f2e5c349c050fbde', 'adapter/adapter_model.safetensors': 'a234860f4ad3788e7b534f815326a02c666d315d268e7e2b82e2f10c937a6684', 'adapter/chat_template.jinja': 'e10ca381b1ccc5cf9db52e371f3b6651576caee0a630b452e2816b2d404d4b65', 'system_prompt.txt': 'ef8908b8c18ecc7614824455aeee8ea7e5c3bf34d2ea9003bdf0f3c4c04b521b', 'label_mapping.json': '3db6b82ff074df7127b8042e7ede69307b22dd8ceb7b6c124a0d95154a536b47'}
asset_dir = Path("/content/cyber_fraud_demo")
for name, expected in ASSET_HASHES.items():
    destination = asset_dir / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    if not destination.exists() or hashlib.sha256(destination.read_bytes()).hexdigest() != expected:
        with urlopen(ASSET_URL + name, timeout=180) as response:
            data = response.read()
        assert hashlib.sha256(data).hexdigest() == expected, f"Download integrity check failed: {name}"
        destination.write_bytes(data)
SYSTEM_PROMPT = (asset_dir / "system_prompt.txt").read_text()
LETTER_TO_LABEL = json.loads((asset_dir / "label_mapping.json").read_text())

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=True)

quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    token=True,
    quantization_config=quantization,
    device_map={"": 0},
    dtype=torch.float16,
)
base_model.eval()

print("Base model loaded successfully.")
import json
from pathlib import Path
from safetensors.torch import load_file
from peft import LoraConfig, get_peft_model
from peft.utils.save_and_load import set_peft_model_state_dict

folder = asset_dir / "adapter"
required = ["adapter_config.json", "adapter_model.safetensors"]
missing = [name for name in required if not (folder / name).exists()]
assert not missing, f"Please upload these files again: {missing}"

config = json.loads((folder / "adapter_config.json").read_text())
assert config["base_model_name_or_path"] == BASE_MODEL

adapter_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=config["r"],
    lora_alpha=config["lora_alpha"],
    lora_dropout=config["lora_dropout"],
    target_modules=config["target_modules"],
    bias=config["bias"],
    inference_mode=True,
)

model = get_peft_model(base_model, adapter_config)

weights = load_file(str(folder / "adapter_model.safetensors"))
weights = {
    key if key.startswith("base_model.model.")
    else "base_model.model." + key: value
    for key, value in weights.items()
}

result = set_peft_model_state_dict(model, weights)
missing_adapter = [key for key in result.missing_keys if "lora_" in key]

assert not missing_adapter, f"Missing adapter weights: {missing_adapter[:5]}"
assert not result.unexpected_keys, f"Unexpected weights: {result.unexpected_keys[:5]}"

model.eval()
print(f"Adapter loaded successfully: {len(weights)} weight tensors.")
tokenizer.chat_template = (asset_dir / "adapter/chat_template.jinja").read_text()


## Enter a complaint
Click **Classify complaint**. The output is a model prediction, not a verified finding. Blank or overly long inputs are rejected. Unexpected model output is flagged for review rather than assigned a category.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from google.colab import output as colab_output
colab_output.enable_custom_widget_manager()

def classify_complaint(text):
    text = text.strip()
    if not text:
        raise ValueError("Please enter a complaint.")
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "Complaint: " + text},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    if inputs["input_ids"].shape[1] > 2048:
        raise ValueError("Please shorten the complaint; this demo accepts up to 2,048 tokens including instructions.")
    inputs = inputs.to(model.device)
    with torch.inference_mode():
        generated = model.generate(**inputs, max_new_tokens=8, do_sample=False,
                                   pad_token_id=tokenizer.eos_token_id)
    raw = tokenizer.decode(generated[0, inputs["input_ids"].shape[1]:],
                           skip_special_tokens=True, clean_up_tokenization_spaces=False).strip()
    return raw, LETTER_TO_LABEL.get(raw)

complaint_box = widgets.Textarea(
    value="A recruiter promised me a job, collected a recruitment fee, then blocked me. The company confirmed that the vacancy was fake.",
    placeholder="Enter a synthetic or anonymized complaint...",
    layout=widgets.Layout(width="100%", height="140px"),
)
classify_button = widgets.Button(description="Classify complaint", button_style="primary",
                                 layout=widgets.Layout(width="200px"))
result_box = widgets.Output()

def on_classify(_):
    classify_button.disabled = True
    try:
        with result_box:
            result_box.clear_output(wait=True)
            print("Classifying…")
            try:
                raw, category = classify_complaint(complaint_box.value)
                result_box.clear_output(wait=True)
                if category is None:
                    print("No valid category returned. Refer for human review.")
                    print("Raw model response:", repr(raw))
                else:
                    print("Predicted category:", raw, "—", category.replace("_", " "))
                    print("This routes an allegation; it does not establish that a crime occurred.")
            except ValueError as error:
                result_box.clear_output(wait=True)
                print(str(error))
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                print("GPU memory is full. Shorten the complaint or restart the session and reload the model.")
    finally:
        classify_button.disabled = False

classify_button.on_click(on_classify)
display(complaint_box, classify_button, result_box)


## Troubleshooting
- **No GPU:** select T4 in runtime settings; free resources may be unavailable.
- **401/403 downloading Llama:** use a read token from the account approved for this model.
- **Library import error after installation:** restart the session and rerun from the first cell.
- **Session ended:** rerun setup and loading. This is a session-based demo, not a persistent hosted service.

The [4-bit loading documentation](https://huggingface.co/docs/transformers/quantization/bitsandbytes) describes the quantization used here.
